# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the FAIR² dataset [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset and show high-level metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSet entities by @id and name
print("Available Record Sets:")
record_sets = meta.record_sets
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs['name']}")

In [ ]:
# Show fields (columns) for each record set
print("\nFields in each Record Set:")
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    for field in rs['fields']:
        print(f"  - @id: {field['@id']}, name: {field['name']}, dataType: {field.get('dataType')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify the record set(s) to extract. For this dataset, typically one main record set exists.
record_set_ids = [rs['@id'] for rs in meta.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\n[INFO] Loaded record set '{record_set_id}' with shape {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distribution, or grouping data by attribute.

In [ ]:
# Choose main record set (change index if more than one is relevant!)
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Pick a numeric field for filtering and normalization
# E.g., field @id: 'accession.coverage' or 'coverage', if available
numeric_field_id = None
group_field_id = None

# Find first numeric field
for rs in meta.record_sets:
    if rs['@id'] == main_record_set_id:
        for f in rs['fields']:
            dtype = f.get('dataType', '').lower()
            if dtype in {'float', 'integer', 'number'} or 'float' in dtype or 'number' in dtype or 'int' in dtype:
                if f['@id'] in df.columns:
                    numeric_field_id = f['@id']
                    break
# For demonstration, unless available, use first
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes('number').columns[0]

# Pick a grouping field (example: 'description' or any categorical)
for rs in meta.record_sets:
    if rs['@id'] == main_record_set_id:
        for f in rs['fields']:
            dtype = f.get('dataType', '').lower()
            if dtype in {'text', 'string'} or 'text' in dtype or 'str' in dtype:
                if f['@id'] in df.columns:
                    group_field_id = f['@id']
                    break

print(f"Numeric field chosen for filtering: {numeric_field_id}")
if group_field_id:
    print(f"Categorical/group field: {group_field_id}")

# Filter records where numeric field > its median
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")

# Normalize the chosen numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional: Grouped statistics
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
    print("\nGrouped mean values by group field:")
    display(grouped.head(8))

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='dodgerblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.tight_layout()
plt.show()

if group_field_id and group_field_id in df.columns:
    # Top 6 categories only
    top_groups = df[group_field_id].value_counts().head(6).index
    plt.figure(figsize=(9,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, filter, and visualize the FAIR² dataset using the Croissant schema and the `mlcroissant` Python library. You can extend this template for advanced statistics, machine learning, or custom biological analyses using the provided dataframe(s) and field `@id`s.